In [1]:
# ── REQUIRED: run this cell first and paste your HuggingFace token when prompted ──
# 1. Accept terms at https://huggingface.co/facebook/sam3
# 2. Create a Read token at https://huggingface.co/settings/tokens
import os
HF_TOKEN = input("Paste your HuggingFace token: ").strip()
os.environ["HF_TOKEN"] = HF_TOKEN
print("HF_TOKEN set" if HF_TOKEN.startswith("hf_") else "⚠ Token looks wrong — double-check it")


HF_TOKEN set


In [2]:
# Cell 1: install TrueRender demo dependencies, SAM 3, and patched TripoSR.
import os
import subprocess
import sys
from pathlib import Path

PYTHON = sys.executable
APP_DIR = Path.cwd()

# Auto-clone (or pull) when running in Colab from the default /content directory.
if not (APP_DIR / "app.py").exists():
    REPO_DIR = Path("/content/TrueRender")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--quiet"], check=True)
    else:
        subprocess.run(["git", "clone", "--quiet", "https://github.com/manny-serrano/TrueRender.git", str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    APP_DIR = Path.cwd()

print("Notebook Python:", PYTHON)
print("Working directory:", APP_DIR)
assert (APP_DIR / "app.py").exists(), "Could not find app.py — repo clone may have failed."

_hf_token = os.environ.get("HF_TOKEN", "")
if not _hf_token:
    raise RuntimeError("Set HF_TOKEN in Cell 0 before running this cell.")
import huggingface_hub as _hfhub
_hfhub.login(token=_hf_token, add_to_git_credential=False)
print("Logged in to HuggingFace")

base_install = [
    PYTHON, "-m", "pip", "install", "-q", "--upgrade",
    "fastapi", "uvicorn[standard]", "python-multipart", "trimesh", "pyngrok",
    "numpy==1.26.4", "opencv-python-headless==4.10.0.84", "pillow<10",
    "transformers==4.42.3",
    "timm>=1.0.17", "iopath>=0.1.10", "ftfy==6.1.1",
    "setuptools>=70",
]
print("Running:", " ".join(base_install))
subprocess.run(base_install, check=True)

# Install SAM 3.
os.chdir("/content")
if not os.path.exists("/content/sam3"):
    subprocess.run(["git", "clone", "--quiet", "https://github.com/facebookresearch/sam3.git"], check=True)
os.chdir("/content/sam3")

# Patch SAM3 model_builder.py: system pkg_resources is Python 3.12-incompatible;
# replace all three usages with os.path (idempotent).
mb_py = Path("/content/sam3/sam3/model_builder.py")
mb_code = mb_py.read_text()
if "import pkg_resources" in mb_code:
    mb_code = mb_code.replace("import pkg_resources\n", "", 1)
    mb_code = mb_code.replace(
        'pkg_resources.resource_filename(\n            "sam3", "assets/bpe_simple_vocab_16e6.txt.gz"\n        )',
        'os.path.join(os.path.dirname(os.path.abspath(__file__)), "assets", "bpe_simple_vocab_16e6.txt.gz")',
    )
    mb_py.write_text(mb_code)
    print("Patched SAM3 model_builder.py: replaced pkg_resources with os.path")

subprocess.run([PYTHON, "-m", "pip", "install", "-e", "."], check=True)
verify = subprocess.run([PYTHON, "-c", "from sam3.model_builder import build_sam3_video_predictor; print('sam3 import OK')"], capture_output=True, text=True)
if verify.returncode != 0:
    raise RuntimeError("SAM 3 import failed after install:\n" + verify.stderr)
print(verify.stdout.strip())
print("SAM 3 ready")

# Install TripoSR clean mesh generator
TRIPOSR_REPO = "/content/TripoSR"
os.environ["TRIPOSR_REPO"] = TRIPOSR_REPO
os.chdir("/content")
if not os.path.exists(TRIPOSR_REPO):
    subprocess.run(["git", "clone", "--quiet", "https://github.com/VAST-AI-Research/TripoSR", TRIPOSR_REPO], check=True)
os.chdir(TRIPOSR_REPO)

install_cmds = [
    [PYTHON, "-m", "pip", "install", "-q", "--upgrade", "setuptools", "wheel"],
    [PYTHON, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
]
for cmd in install_cmds:
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

# Restore files before patching so this cell is idempotent even after a failed previous patch.
subprocess.run(["git", "checkout", "--", "run.py", "tsr/utils.py"], check=True)

# Patch TripoSR so --no-remove-bg does not import rembg/onnxruntime at module load time.
# We already provide a SAM-masked object composited onto gray, so rembg is unnecessary.
run_py = Path(TRIPOSR_REPO) / "run.py"
code = run_py.read_text()
code = code.replace(
    "import rembg\n",
    "# TRUERENDER_PATCH_LAZY_REMBG_RUN: rembg is imported lazily only when needed\nrembg = None\n",
    1,
)
expected = "if args.no_remove_bg:\n    rembg_session = None\nelse:\n    rembg_session = rembg.new_session()"
patched = "if args.no_remove_bg:\n    rembg_session = None\nelse:\n    import rembg\n    rembg_session = rembg.new_session()"
if expected not in code:
    raise RuntimeError("Could not find expected rembg_session block in TripoSR run.py")
code = code.replace(expected, patched, 1)

expected = "out_mesh_path = os.path.join(output_dir, str(i), f\"mesh.{args.model_save_format}\")"
patched = "out_mesh_path = os.path.join(output_dir, str(i), f\"mesh.{args.model_save_format}\")\nos.makedirs(os.path.dirname(out_mesh_path), exist_ok=True)"
if expected not in code:
    raise RuntimeError("Could not find expected out_mesh_path line in TripoSR run.py")
code = code.replace(expected, patched, 1)

run_py.write_text(code)
print("Patched TripoSR run.py for lazy rembg import and robust export directory creation")

utils_py = Path(TRIPOSR_REPO) / "tsr" / "utils.py"
code = utils_py.read_text()
code = code.replace(
    "import rembg\n",
    "# TRUERENDER_PATCH_LAZY_REMBG_UTILS: rembg is imported lazily only when remove_background is called\nrembg = None\n",
    1,
)
expected = "    if do_remove:\n        image = rembg.remove(image, session=rembg_session, **rembg_kwargs)"
patched = "    if do_remove:\n        import rembg\n        image = rembg.remove(image, session=rembg_session, **rembg_kwargs)"
if expected not in code:
    raise RuntimeError("Could not find expected remove_background block in TripoSR tsr/utils.py")
code = code.replace(expected, patched, 1)
utils_py.write_text(code)
print("Patched TripoSR tsr/utils.py to avoid rembg import when unused")

probe = subprocess.run(
    [PYTHON, "-c", "import sys; import tsr.utils; print(sys.executable); print('tsr.utils import ok')"],
    capture_output=True,
    text=True,
    check=True,
    cwd=TRIPOSR_REPO,
)
print("Child Python probe:\n" + probe.stdout)

os.chdir(APP_DIR)

# Make SAM3 importable in the running kernel process (editable-install .pth files are
# processed at interpreter startup, so pip install -e . doesn't update a live kernel).
if "/content/sam3" not in sys.path:
    sys.path.insert(0, "/content/sam3")

print("TripoSR installed")
print("TrueRender demo setup complete")

Notebook Python: /usr/bin/python3
Working directory: /content/TrueRender
Token will not been saved to git credential helper. Pass `add_to_git_credential=True` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /root/.cache/huggingface/token
Login successful
Logged in to HuggingFace
Running: /usr/bin/python3 -m pip install -q --upgrade fastapi uvicorn[standard] python-multipart trimesh pyngrok numpy==1.26.4 opencv-python-headless==4.10.0.84 pillow<10 transformers==4.42.3 timm>=1.0.17 iopath>=0.1.10 ftfy==6.1.1 setuptools>=70
sam3 import OK
SAM 3 ready
Running: /usr/bin/python3 -m pip install -q --upgrade setuptools wheel
Running: /usr/bin/python3 -m pip install -q -r requirements.txt
Patched TripoSR run.py for lazy rembg import and robust export directory creation
Patched TripoSR tsr/utils.py to avoid rembg import when unused
Child Python probe:
/usr/bin/python3
tsr.utils import ok

TripoSR installed
TrueRender demo setu

In [ ]:
# Cell 2: launch FastAPI and expose it with cloudflared (no account needed).
import os
import re
import subprocess
import threading
import time

import uvicorn

APP_DIR = os.getcwd()
os.environ.setdefault("TRIPOSR_REPO", "/content/TripoSR")

def run_server():
    uvicorn.run("app:app", host="0.0.0.0", port=8000, log_level="info")

if not globals().get("_truerender_server_thread") or not _truerender_server_thread.is_alive():
    _truerender_server_thread = threading.Thread(target=run_server, daemon=True)
    _truerender_server_thread.start()
    time.sleep(2)

# Install cloudflared if not already present.
if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "/usr/local/bin/cloudflared",
    ], check=True)
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)

# Kill any previous tunnel process so we get a fresh URL.
if globals().get("_cf_proc") and _cf_proc.poll() is None:
    _cf_proc.terminate()
    time.sleep(1)

# Start tunnel; merge stderr into stdout so nothing is missed.
_cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
public_url = None
for _line in _cf_proc.stdout:
    print(_line, end="", flush=True)
    if public_url is None:
        _m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', _line)
        if _m:
            public_url = _m.group()
            break

time.sleep(3)  # let Cloudflare's edge propagate the tunnel
print("\nOpen this URL in a new tab to use the demo")
print(public_url)


INFO:     Started server process [8891]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


2026-04-26T16:54:41Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-04-26T16:54:41Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-04-26T16:54:45Z INF +--------------------------------------------------------------------------------------------+
2026-04-26T16:54:45Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-04-26T16:54:45Z INF |  https://posters-humanity-talent-snap.trycloudflare.co

INFO:     152.3.43.52:0 - "GET / HTTP/1.1" 200 OK
INFO:     152.3.43.52:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     152.3.43.52:0 - "POST /reconstruct/image HTTP/1.1" 200 OK
Patched TripoSR run.py for lazy rembg import and robust export directory creation
Patched TripoSR tsr/utils.py to avoid rembg import when unused
INFO:     152.3.43.52:0 - "GET /jobs/d8509e5d-897e-48d1-9b8b-3927e8bacd63 HTTP/1.1" 200 OK
INFO:     152.3.43.52:0 - "GET /jobs/d8509e5d-897e-48d1-9b8b-3927e8bacd63 HTTP/1.1" 200 OK
Child Python probe:
/usr/bin/python3
tsr.utils import ok

INFO:     152.3.43.52:0 - "GET /jobs/d8509e5d-897e-48d1-9b8b-3927e8bacd63 HTTP/1.1" 200 OK
INFO:     152.3.43.52:0 - "GET /jobs/d8509e5d-897e-48d1-9b8b-3927e8bacd63 HTTP/1.1" 200 OK
INFO:     152.3.43.52:0 - "GET /jobs/d8509e5d-897e-48d1-9b8b-3927e8bacd63 HTTP/1.1" 200 OK


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
INFO 2026-04-26 16:56:40,120 8891 sam3_video_predictor.py: 109: using the following GPU IDs: [0]
INFO 2026-04-26 16:56:40,365 8891 sam3_video_predictor.py: 125: 


	*** START loading model on all ranks ***


INFO 2026-04-26 16:56:40,366 8891 sam3_video_predictor.py: 127: loading model on rank=0 with world_size=1 -- this could take a while ...


INFO:     152.3.43.52:0 - "GET /jobs/d8509e5d-897e-48d1-9b8b-3927e8bacd63 HTTP/1.1" 200 OK
INFO:     152.3.43.52:0 - "GET /jobs/d8509e5d-897e-48d1-9b8b-3927e8bacd63 HTTP/1.1" 200 OK
INFO:     152.3.43.52:0 - "GET /jobs/d8509e5d-897e-48d1-9b8b-3927e8bacd63 HTTP/1.1" 200 OK
INFO:     152.3.43.52:0 - "GET /jobs/d8509e5d-897e-48d1-9b8b-3927e8bacd63 HTTP/1.1" 200 OK
INFO:     152.3.43.52:0 - "GET /jobs/d8509e5d-897e-48d1-9b8b-3927e8bacd63 HTTP/1.1" 200 OK
INFO:     152.3.43.52:0 - "GET /jobs/d8509e5d-897e-48d1-9b8b-3927e8bacd63 HTTP/1.1" 200 OK


...

frame loading (image folder) [rank=0]: 100%|██████████| 1/1 [00:00<00:00,  5.28it/s]
INFO 2026-04-26 16:56:53,670 8891 sam3_base_predictor.py: 134: started new session 6cb47f19-86e8-4eea-9019-09446ac901e6


INFO:     152.3.43.52:0 - "GET /jobs/d8509e5d-897e-48d1-9b8b-3927e8bacd63 HTTP/1.1" 200 OK
INFO:     152.3.43.52:0 - "GET /jobs/d8509e5d-897e-48d1-9b8b-3927e8bacd63 HTTP/1.1" 200 OK


propagate_in_video:   0%|          | 0/1 [00:00<?, ?it/s]

INFO 2026-04-26 16:56:56,820 8891 sam3_base_predictor.py: 293: propagation ended in session 6cb47f19-86e8-4eea-9019-09446ac901e6
INFO 2026-04-26 16:56:57,033 8891 sam3_base_predictor.py: 312: removed session 6cb47f19-86e8-4eea-9019-09446ac901e6


INFO:     152.3.43.52:0 - "GET /jobs/d8509e5d-897e-48d1-9b8b-3927e8bacd63 HTTP/1.1" 200 OK
INFO:     152.3.43.52:0 - "GET /jobs/d8509e5d-897e-48d1-9b8b-3927e8bacd63 HTTP/1.1" 200 OK
INFO:     152.3.43.52:0 - "GET /jobs/d8509e5d-897e-48d1-9b8b-3927e8bacd63 HTTP/1.1" 200 OK
INFO:     152.3.43.52:0 - "GET /jobs/d8509e5d-897e-48d1-9b8b-3927e8bacd63 HTTP/1.1" 200 OK
Saved SAM outputs for 1 frames
Best canonical frame: 0 score= 1.379 {'area_frac': 0.08537249294007726, 'bbox_area_frac': 0.10629251700680271, 'fill_ratio': 0.803184413580247, 'center_penalty': 0.012276785714285698}
INFO:     152.3.43.52:0 - "GET /jobs/d8509e5d-897e-48d1-9b8b-3927e8bacd63 HTTP/1.1" 200 OK
Saved canonical input: /content/TrueRender/outputs/d8509e5d-897e-48d1-9b8b-3927e8bacd63/canonical/canonical_rgba.png
Child Python probe:
/usr/bin/python3
child python ok

Saved TripoSR input: /content/TrueRender/outputs/d8509e5d-897e-48d1-9b8b-3927e8bacd63/triposr_gray_bg.png
Running: /usr/bin/python3 -u run.py /content/TrueRend